In [ ]:
# Colab setup -- installs SoftMobility when running on Google Colab.
# Safe to run locally: it does nothing outside Colab.
try:
    import google.colab  # noqa: F401
    %pip install -q git+https://github.com/celoy/SoftMobility.git
except ImportError:
    pass

# Tutorial 21 — Jeffery orbit of a deformable body

Tutorial **14** reproduced Jeffery's tumbling orbit for a *rigid*
two-sphere dumbbell in pure shear. Here we revisit the same flow with a
**deformable** dumbbell: the half-distance between the two spheres is a
degree of freedom `d` paired with a linear Hookean spring of stiffness
`k`. As the body sweeps through the extensional and compressive
quadrants of the shear flow, the spring stretches and contracts.

* In the rigid limit (`k → ∞`) the orbit recovers the classic Jeffery
  period and `d ≈ 0` throughout.
* For finite `k` the stretching DOF oscillates at twice the tumbling
  frequency (extension-compression-extension per half-tumble) and
  modulates the body shape.

We compare a stiff and a floppy version side by side.


In [1]:
import jax.numpy as jnp
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import softmobility as sm

np.set_printoptions(precision=3, suppress=True)


## Body and flow

Two spheres at body-frame positions `±(length + d/2)` along $\boldsymbol{E}_1$. The
spring force on each sphere points toward the body centre and is linear
in `d`.

The shear flow `u = (γ̇·z, 0, 0)` has its plane in xz so the body's long
axis stays in the bending plane.


In [2]:
yaml_template = """
dof_names:    [d]
design_names: [k, length]

defaults:
  d0:     0.0
  k:      {k}
  length: 1.0

spheres:
  - radius: 0.5
    position: [-length - d0 / 2, 0, 0]
    force:    [k * d0, 0, 0]
  - radius: 0.5
    position: [length + d0 / 2, 0, 0]
    force:    [-k * d0, 0, 0]
"""


def make_body_and_rollout(k_value):
    body = sm.SoftBody(yaml_template.format(k=k_value), verbose=False)
    shear_xz = sm.Flow(lambda pos, t: jnp.array([pos[2], 0.0, 0.0]))  # γ̇ = 1
    rollout = sm.FlowBodyRollout(soft_body=body, flow=shear_xz)
    return body, rollout


body_stiff, roll_stiff = make_body_and_rollout(k_value=100.0)
body_soft,  roll_soft  = make_body_and_rollout(k_value=2.0)
print(repr(body_soft))


SPHERE ASSEMBLY
  2 spheres
  1 degrees of freedom
  2 design parameters
  0 input parameters

Default values
  degrees of freedom dof: ['d0'] = [0.]
  design parameters param: ['k', 'length'] = [2. 1.]
  input parameters param: []

SPHERE 0
  radius: 0.5
  position: [-1.  0.  0.]
  orientation: [0. 0. 0.]
  C_H:
[]
  C_K:
[[2.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]

SPHERE 1
  radius: 0.5
  position: [1. 0. 0.]
  orientation: [0. 0. 0.]
  C_H:
[]
  C_K:
[[-2.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]]



## Run the rollouts

Initial body orientation tilted 45° around $\boldsymbol{E}_2$ so the body starts on
the extensional axis of the strain field; we then watch it sweep
through several tumble periods.


In [3]:
init_orientation = jnp.array([0.0, np.pi / 4, 0.0])
dt = 0.02
n_steps = 1500   # ≈ 5 Jeffery periods of an aspect-ratio-2 body

pos_s, ori_s, dofs_s = roll_stiff.rollout(dt=dt, n_steps=n_steps, init_orientation=init_orientation)
pos_f, ori_f, dofs_f = roll_soft.rollout(dt=dt, n_steps=n_steps, init_orientation=init_orientation)
pos_s.block_until_ready()
pos_f.block_until_ready()

print(f"max |d| over the trajectory  (stiff k=100): {float(jnp.max(jnp.abs(dofs_s))):.3e}")
print(f"max |d| over the trajectory  (soft  k=2)  : {float(jnp.max(jnp.abs(dofs_f))):.3e}")


max |d| over the trajectory  (stiff k=100): 7.439e-02
max |d| over the trajectory  (soft  k=2)  : 2.087e+00


## Compare orbits

Top row: body orientation around $\boldsymbol{E}_2$ over time. Both trajectories
tumble at the Jeffery rate; the soft body lags slightly because the
flexible bond spends part of the strain work on stretching instead of
rotation.

Bottom row: the stretching DOF `d(t)`. The stiff body's DOF stays close
to zero, while the soft body's DOF oscillates at twice the tumbling
frequency — extension when the body axis aligns with the extensional
axis of the strain, compression when it aligns with the compressive
axis.


In [4]:
t = np.arange(n_steps) * dt

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Body angle Θ_y(t)  —  stiff (k=100)",
        "Body angle Θ_y(t)  —  soft (k=2)",
        "DOF d(t)            —  stiff",
        "DOF d(t)            —  soft",
    ),
    shared_xaxes=True,
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
)
fig.add_trace(go.Scatter(x=t, y=np.asarray(ori_s[:, 1]), mode="lines",
                         line=dict(color="#1f77b4"), showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=np.asarray(ori_f[:, 1]), mode="lines",
                         line=dict(color="#d62728"), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=t, y=np.asarray(dofs_s[:, 0]), mode="lines",
                         line=dict(color="#1f77b4"), showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=np.asarray(dofs_f[:, 0]), mode="lines",
                         line=dict(color="#d62728"), showlegend=False), row=2, col=2)
for col in (1, 2):
    fig.update_xaxes(title_text="time", row=2, col=col)
fig.update_yaxes(title_text="Θ_y  [rad]", row=1, col=1)
fig.update_yaxes(title_text="Θ_y  [rad]", row=1, col=2)
fig.update_yaxes(title_text="d  [length]", row=2, col=1)
fig.update_yaxes(title_text="d  [length]", row=2, col=2)
fig.update_layout(width=950, height=600, plot_bgcolor="white")
fig.show()


## Notes

* The stretching DOF feeds back into the **bead positions** (each sphere
  shifts by `±d/2`), so it couples directly to the ambient strain — no
  redundancy with rigid-body rotation.
* For `k → ∞` the bond becomes inextensible and the dynamics reduce
  to the rigid Jeffery orbit of tutorial **14**.
* For very low `k` the bond fully extends and contracts each tumble,
  producing the classic stretch-coil instability when combined with
  bending. See tutorial **12 — flexible fiber 2D** for a multi-bead
  fiber where bending and stretching co-exist.
